# Graph ML Overview — Exercises

Companion to the note [Graph ML Overview](Graph%20ML%20Overview.md).

Practise the basic objects of graph ML ($A$, $D$, $L = D - A$), what they encode (walks, cuts, components, random walks), the classic hand-crafted features (degree, clustering, closeness, PageRank), spectral community detection, the Weisfeiler–Lehman test behind graph kernels and GNN expressiveness, and a DeepWalk-style shallow embedding. Graphs are plain NumPy adjacency matrices; no graph library needed.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.sparse.csgraph import shortest_path
from sklearn.cluster import KMeans
rng = np.random.default_rng(0)

def from_edges(n, edges):
    """Undirected adjacency matrix from an edge list."""
    A = np.zeros((n, n))
    for i, j in edges:
        A[i, j] = A[j, i] = 1
    return A

# Two triangles joined by the bridge 2-3 (used in the by-hand exercises)
E_small = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
A_small = from_edges(6, E_small)

def sbm(sizes, p_in, p_out, seed=0):
    """Stochastic block model: returns (A, labels)."""
    r = np.random.default_rng(seed)
    labels = np.repeat(np.arange(len(sizes)), sizes)
    Pm = np.where(labels[:, None] == labels[None, :], p_in, p_out)
    U = np.triu(r.random(Pm.shape) < Pm, 1)
    return (U + U.T).astype(float), labels

A_sbm, y_sbm = sbm([50, 50], 0.3, 0.02)

## Part A · Concepts

### Exercise 1 · Which task is it?
*🧠 Concept · ★☆☆*

Classify each problem as **node classification**, **link prediction**, **graph classification/regression** or **community detection**:
1. Predict whether a molecule is toxic.
2. Recommend new friends on a social network.
3. Flag fraudulent accounts in a transaction network, given some labelled accounts.
4. Find groups of densely connected proteins with no labels at all.
5. Predict the solubility (a real number) of a molecule.
6. Predict which drug pairs will have side effects when taken together.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: what is the unit that receives a prediction? A node, a pair of nodes, or a whole graph? Are there labels?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Graph classification · 2. Link prediction ([[Link Prediction]]) · 3. Node classification (semi-supervised, [[Graph Neural Networks]]) ·
4. Community detection (unsupervised, [[Clustering]]) · 5. Graph regression · 6. Link prediction (edges between drug nodes, possibly typed).

</details>

### Exercise 2 · Permutation invariance
*🧠 Concept · ★★☆*

The adjacency matrix depends on how we number the nodes: relabelling with a permutation matrix $P$ gives $PAP^\top$.
(a) Why can't we simply flatten $A$ and feed it to an MLP for graph classification?
(b) Which of these functions are permutation **invariant**: $\operatorname{tr}(A)$, $\mathbf 1^\top A\mathbf 1$, $A_{12}$, the sorted degree sequence,
the eigenvalues of $L$? (c) What does **equivariant** mean for a node-level function $f(A, X)$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A function $g$ is invariant if $g(PAP^\top) = g(A)$ for every permutation $P$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The same graph has up to $n!$ different matrices; an MLP would treat them as different inputs, it needs fixed size $n$, and it cannot generalise across sizes.
(b) Invariant: $\operatorname{tr}(A)$, $\mathbf 1^\top A\mathbf 1$ (= $2|E|$), sorted degrees, eigenvalues of $L$ (since $PLP^\top$ is similar to $L$). **Not** invariant: $A_{12}$.
(c) Equivariant: relabelling the input relabels the output the same way, $f(PAP^\top, PX) = Pf(A,X)$. Message-passing GNNs are equivariant at the node level and become invariant after a sum/mean readout.

</details>

### Exercise 3 · Transductive vs inductive
*🧠 Concept · ★★☆*

The note lists four approach families: hand-crafted features, graph kernels, shallow embeddings (DeepWalk, node2vec, MF) and GNNs.
(a) Shallow embeddings learn a free vector $z_v$ per node. What happens when a new user joins the network tomorrow?
(b) Which families can use node **attributes** (e.g. a user's age, a paper's text)?
(c) Graph kernels and GNNs are both used for graph classification. Name one advantage of each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A lookup table $Z\in\mathbb R^{|V|\times d}$ has no row for an unseen node.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Shallow embeddings are **transductive**: there is no embedding for an unseen node without re-training (or extra optimisation). GNNs are **inductive**: they compute
$h_v$ from features and neighbours with shared weights, so they apply to new nodes and new graphs.
(b) Hand-crafted features can be concatenated with attributes; GNNs use them natively as $X$. Plain DeepWalk/node2vec ignore attributes; many graph kernels only use labels.
(c) Kernels (e.g. WL): no training of the representation, strong on small datasets, convex SVM on top ([[Kernel Methods]]). GNNs: learn task-specific features end-to-end, use continuous attributes, scale with mini-batching.

</details>

## Part B · By hand

### Exercise 4 · Adjacency, degree and Laplacian
*✍️ By hand · ★☆☆*

The graph `A_small` has 6 nodes and edges $\{0,1\},\{0,2\},\{1,2\},\{2,3\},\{3,4\},\{3,5\},\{4,5\}$ (two triangles joined by a bridge).
Write down the degree vector $d$ and the row of $L = D - A$ that belongs to node 2. What is $\operatorname{tr}(L)$?

In [ ]:
d_hand = None      # degrees of nodes 0..5
L_row2 = None      # row 2 of L
trace_L = None

check('degrees', d_hand, A_small.sum(1))
check('row 2 of L', L_row2, (np.diag(A_small.sum(1)) - A_small)[2])
check('trace', trace_L, np.trace(np.diag(A_small.sum(1)) - A_small))

<details>
<summary><b>💡 Hint</b></summary>

Node 2 is adjacent to 0, 1 and 3. The diagonal of $L$ holds the degrees, off-diagonals are $-1$ for edges.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$d = (2, 2, 3, 3, 2, 2)$. Row 2 of $L$: $(-1, -1, 3, -1, 0, 0)$. $\operatorname{tr}(L) = \sum_i d_i = 2|E| = 14$.
Every row of $L$ sums to zero, i.e. $L\mathbf 1 = 0$.

```python
d_hand = [2, 2, 3, 3, 2, 2]
L_row2 = [-1, -1, 3, -1, 0, 0]
trace_L = 14
```

</details>

### Exercise 5 · The Laplacian quadratic form
*✍️ By hand · ★★☆*

(a) Prove that for an undirected graph $x^\top L x = \sum_{\{i,j\}\in E}(x_i - x_j)^2$. Conclude that $L$ is positive semi-definite.
(b) For `A_small` and $x = (1,1,1,-1,-1,-1)$ compute $x^\top L x$ by counting edges. How does this relate to the **cut** between the two triangles?

In [ ]:
quad_form = None

_x = np.array([1, 1, 1, -1, -1, -1.])
check('x^T L x', quad_form, _x @ (np.diag(A_small.sum(1)) - A_small) @ _x)

<details>
<summary><b>💡 Hint</b></summary>

$x^\top Dx = \sum_i d_ix_i^2$ and $x^\top Ax = 2\sum_{\{i,j\}\in E}x_ix_j$. Combine and complete the square edge by edge.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $x^\top Lx = \sum_i d_ix_i^2 - 2\sum_{E}x_ix_j = \sum_{\{i,j\}\in E}(x_i^2 + x_j^2 - 2x_ix_j) = \sum_E (x_i-x_j)^2 \ge 0$, since each edge contributes $x_i^2$ to $d_i x_i^2$ and $x_j^2$ to $d_jx_j^2$.

(b) Only the bridge $\{2,3\}$ joins nodes with different signs: $(1-(-1))^2 = 4$. For an indicator $x\in\{\pm1\}^n$, $x^\top Lx = 4\cdot\text{cut}(S,\bar S)$.
Minimising $x^\top Lx$ over real $x\perp\mathbf 1$ with $\lVert x\rVert=1$ is the relaxation that gives **spectral clustering** (Exercise 13): the minimiser is the Fiedler vector.

```python
quad_form = 4.0
```

</details>

### Exercise 6 · Counting walks and triangles
*✍️ By hand · ★★☆*

$(A^k)_{ij}$ counts walks of length $k$ from $i$ to $j$. For `A_small`, find by reasoning (no matrix multiplication):
(a) $(A^2)_{00}$, (b) $(A^2)_{03}$, (c) the number of triangles via $\operatorname{tr}(A^3)/6$. Why divide by 6?

In [ ]:
A2_00 = None
A2_03 = None
n_triangles = None

_A2 = A_small @ A_small
check('(A^2)_00', A2_00, _A2[0, 0])
check('(A^2)_03', A2_03, _A2[0, 3])
check('triangles', n_triangles, np.trace(_A2 @ A_small) / 6)

<details>
<summary><b>💡 Hint</b></summary>

$(A^2)_{ii} = d_i$ and $(A^2)_{ij} = |N(i)\cap N(j)|$ (common neighbours, see [[Link Prediction]]).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(A^2)_{00} = d_0 = 2$ (go to a neighbour and back). (b) $(A^2)_{03} = |N(0)\cap N(3)| = |\{1,2\}\cap\{2,4,5\}| = 1$.
(c) Each triangle gives closed 3-walks starting at each of its 3 nodes in 2 directions: $3\cdot 2 = 6$ walks. $\operatorname{tr}(A^3) = 12$, so 2 triangles.

```python
A2_00 = 2
A2_03 = 1
n_triangles = 2
```

</details>

### Exercise 7 · Connected components and the zero eigenvalue
*✍️ By hand · ★★☆*

(a) Using Exercise 5, show that $Lx = 0$ iff $x$ is constant on each connected component.
(b) Conclude that the multiplicity of the eigenvalue 0 of $L$ equals the number of connected components.
(c) If we delete the bridge $\{2,3\}$ from `A_small`, how many zero eigenvalues does $L$ have?

In [ ]:
n_zero_eigs = None

_A = A_small.copy(); _A[2, 3] = _A[3, 2] = 0
check('zero eigenvalues', n_zero_eigs, np.sum(np.abs(np.linalg.eigvalsh(np.diag(_A.sum(1)) - _A)) < 1e-9))

<details>
<summary><b>💡 Hint</b></summary>

$Lx = 0 \Rightarrow x^\top L x = 0 \Rightarrow (x_i - x_j)^2 = 0$ for every edge.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) If $Lx = 0$ then $0 = x^\top Lx = \sum_E(x_i-x_j)^2$, so $x_i = x_j$ along every edge, hence constant on components. Conversely such $x$ satisfies $Lx=0$ (rows sum to zero within each block).
(b) The indicator vectors of the $c$ components form a basis of that null space, so $\dim\ker L = c$.
(c) Two triangles → **2** zero eigenvalues. With the bridge present, the second-smallest eigenvalue $\lambda_2\approx 0.438$ is small but nonzero: a small $\lambda_2$ (algebraic connectivity) signals a near-disconnection, which spectral clustering exploits.

```python
n_zero_eigs = 2
```

</details>

### Exercise 8 · Random walks prefer high-degree nodes
*✍️ By hand · ★★☆*

A simple random walk on an undirected connected graph moves from $i$ to a uniformly random neighbour: $T = D^{-1}A$.
Show that $\pi_i = d_i/(2|E|)$ is a stationary distribution ($\pi^\top T = \pi^\top$). What is $\pi_2$ for `A_small`?
Why is this relevant for DeepWalk and for PageRank?

In [ ]:
pi_2 = None

_T = A_small / A_small.sum(1, keepdims=True)
_w, _V = np.linalg.eig(_T.T); _v = np.real(_V[:, np.argmin(np.abs(_w - 1))]); _v /= _v.sum()
check('pi_2', pi_2, _v[2])

<details>
<summary><b>💡 Hint</b></summary>

$(\pi^\top T)_j = \sum_i \pi_i T_{ij} = \sum_{i\in N(j)}\frac{d_i}{2|E|}\cdot\frac1{d_i}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(\pi^\top T)_j = \sum_{i\in N(j)}\frac{d_i}{2|E|}\frac{1}{d_i} = \frac{d_j}{2|E|} = \pi_j$ ✔. For `A_small`: $\pi_2 = 3/14 \approx 0.214$.

DeepWalk's random walks visit hubs proportionally more often, so hubs dominate the co-occurrence statistics (word2vec's negative sampling partly corrects this).
PageRank on an undirected graph without teleportation would be just degree centrality; teleportation and edge direction make it more interesting.

```python
pi_2 = 3 / 14
```

</details>

### Exercise 9 · PageRank by hand
*✍️ By hand · ★★★*

Directed graph on 3 pages: $0\to1$, $1\to2$, $2\to0$, $2\to1$. With damping $\alpha = 0.85$ PageRank solves
$$p_i = \frac{1-\alpha}{N} + \alpha\sum_{j\to i}\frac{p_j}{d^{\text{out}}_j}.$$
Write the three equations and solve them. Which page ranks highest and why?

In [ ]:
pr_hand = None  # [p0, p1, p2]

_M = np.array([[0, 0, .5], [1, 0, .5], [0, 1, 0]])
check('PageRank', pr_hand, np.linalg.solve(np.eye(3) - 0.85 * _M, np.full(3, 0.05)), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$p_0 = 0.05 + 0.425p_2$, $p_1 = 0.05 + 0.85p_0 + 0.425p_2$, $p_2 = 0.05 + 0.85p_1$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Substitute $p_0$ into $p_1$ to get $p_1 = 0.0925 + 0.78625\,p_2$, then use the $p_2$ equation.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p_1 = 0.05 + 0.85(0.05 + 0.425p_2) + 0.425p_2 = 0.0925 + 0.78625p_2$.
$p_2 = 0.05 + 0.85p_1 = 0.128625 + 0.6683125\,p_2 \Rightarrow p_2 = 0.128625/0.3316875 \approx 0.3878$.
Then $p_1 \approx 0.3974$, $p_0 = 0.05 + 0.425p_2 \approx 0.2148$; the sum is 1 ✔.

Page 1 ranks highest: it receives *all* of page 0's vote plus half of page 2's. Page 2 is close behind because it gets the entire vote of the
high-ranked page 1. Page 0 only gets half of page 2's vote.

```python
pr_hand = [0.2148, 0.3974, 0.3878]
```

</details>

## Part C · Code: features, spectra, WL and embeddings

### Exercise 10 · Degree and clustering coefficient
*💻 Code · ★☆☆*

Implement `clustering(A)` returning the local clustering coefficient $C_i = \frac{2T_i}{d_i(d_i-1)}$ (0 if $d_i<2$), where $T_i$ is the number
of triangles through $i$. Use the matrix identity $T_i = (A^3)_{ii}/2$. Apply it to `A_sbm` and compare the average within the SBM with an
Erdős–Rényi graph of the same density.

In [ ]:
def clustering(A):
    # TODO
    return None

C_sbm = clustering(A_sbm)

def _brute(A):
    n = len(A); C = np.zeros(n)
    for i in range(n):
        nb_ = np.flatnonzero(A[i]); k = len(nb_)
        if k >= 2:
            links = sum(A[u, v] for a_, u in enumerate(nb_) for v in nb_[a_ + 1:])
            C[i] = links / (k * (k - 1) / 2)
    return C
check('clustering coefficients', C_sbm, _brute(A_sbm))
check('small graph', None if C_sbm is None else clustering(A_small), [1, 1, 1/3, 1/3, 1, 1])

<details>
<summary><b>💡 Hint</b></summary>

`T = np.diag(A @ A @ A) / 2`; guard against division by zero with `np.where(d >= 2, ..., 0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The SBM's average clustering (≈ 0.28) is far higher than its overall edge density (≈ 0.16): within a community, two neighbours of a node are
likely to be connected. An Erdős–Rényi graph has $\mathbb E[C_i]\approx p$, so high clustering is a signature of community structure (and triadic closure,
the intuition behind common-neighbour scores in [[Link Prediction]]).

```python
def clustering(A):
    d = A.sum(1)
    T = np.diag(A @ A @ A) / 2
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(d >= 2, 2 * T / (d * (d - 1)), 0.0)

C_sbm = clustering(A_sbm)
dens = A_sbm.sum() / (len(A_sbm) * (len(A_sbm) - 1))
print('avg clustering:', C_sbm.mean().round(3), ' edge density:', dens.round(3))
```

</details>

### Exercise 11 · BFS and closeness centrality
*💻 Code · ★★☆*

Implement `bfs_dist(A, s)` (breadth-first search, unweighted) and use it for closeness centrality
$c(v) = \frac{n-1}{\sum_u \operatorname{dist}(v,u)}$ on `A_small`. Which nodes are most central?

In [ ]:
from collections import deque
def bfs_dist(A, s):
    # TODO: return array of distances from s (np.inf if unreachable)
    return None

closeness = None  # array of length 6

_D = shortest_path(A_small, unweighted=True)
check('closeness', closeness, (len(A_small) - 1) / _D.sum(1))

<details>
<summary><b>💡 Hint</b></summary>

Keep a queue; when popping $u$, for each neighbour $w$ with `dist[w] == inf` set `dist[w] = dist[u] + 1` and enqueue it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bridge nodes 2 and 3 have closeness $5/7\approx0.714$; the others $5/10 = 0.5$. Degree alone (3 vs 2) agrees here, but closeness and betweenness
capture *global* position: a bridge with low degree can still be very central.

```python
from collections import deque
def bfs_dist(A, s):
    dist = np.full(len(A), np.inf); dist[s] = 0
    q = deque([s])
    while q:
        u = q.popleft()
        for w in np.flatnonzero(A[u]):
            if dist[w] == np.inf:
                dist[w] = dist[u] + 1
                q.append(w)
    return dist

closeness = np.array([(len(A_small) - 1) / bfs_dist(A_small, v).sum() for v in range(len(A_small))])
print(np.round(closeness, 3))
```

</details>

### Exercise 12 · PageRank by power iteration
*💻 Code · ★★☆*

Implement `pagerank(Adir, alpha=0.85)` for a **directed** adjacency matrix (`Adir[i, j] = 1` for $i\to j$) by power iteration.
Treat **dangling** nodes (no out-links) as linking to every node. Test it on the random directed graph below.

In [ ]:
Adir = (rng.random((30, 30)) < 0.08).astype(float)
np.fill_diagonal(Adir, 0)
Adir[5] = 0  # make node 5 dangling

def pagerank(Adir, alpha=0.85, tol=1e-12):
    # TODO
    return None

pr = pagerank(Adir)

_n = len(Adir); _out = Adir.sum(1)
_T = np.where(_out[:, None] > 0, Adir / np.maximum(_out, 1)[:, None], 1 / _n)   # row-stochastic
_G = 0.85 * _T + 0.15 / _n
_w, _V = np.linalg.eig(_G.T); _v = np.real(_V[:, np.argmax(np.real(_w))]); _v /= _v.sum()
check('PageRank (vs leading eigenvector of the Google matrix)', pr, _v, tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

Build a row-stochastic $T$ (dangling rows = $1/n$), then iterate $p\leftarrow\alpha T^\top p + (1-\alpha)/n$ until the change is tiny.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Google matrix $G = \alpha T + \frac{1-\alpha}{n}\mathbf 1\mathbf 1^\top$ is positive and stochastic, so by Perron–Frobenius it has a unique stationary
distribution, and power iteration converges at rate $\alpha$ (here about 0.85 per step, so ~170 iterations for $10^{-12}$).

```python
Adir = (rng.random((30, 30)) < 0.08).astype(float)
np.fill_diagonal(Adir, 0)
Adir[5] = 0

def pagerank(Adir, alpha=0.85, tol=1e-12):
    n = len(Adir); out = Adir.sum(1)
    T = np.where(out[:, None] > 0, Adir / np.maximum(out, 1)[:, None], 1 / n)
    p = np.full(n, 1 / n)
    while True:
        p_new = alpha * T.T @ p + (1 - alpha) / n
        if np.abs(p_new - p).sum() < tol:
            return p_new
        p = p_new

pr = pagerank(Adir)
print('top pages:', np.argsort(-pr)[:5])
```

</details>

### Exercise 13 · Spectral community detection
*💻 Code · ★★☆*

Recover the two communities of `A_sbm` with the **Fiedler vector** (eigenvector of the second-smallest eigenvalue of $L$): assign nodes by its sign.
Store the predicted labels in `y_spec` and the accuracy (allowing for swapped labels) in `acc_spec`.

In [ ]:
y_spec = None
acc_spec = None

check('accuracy >= 0.95', None if acc_spec is None else acc_spec >= 0.95, True)

<details>
<summary><b>💡 Hint</b></summary>

`w, V = np.linalg.eigh(L)` returns eigenvalues in ascending order; the Fiedler vector is `V[:, 1]`. Accuracy: `max(mean(y==y_sbm), mean(y!=y_sbm))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Fiedler vector is the relaxed minimiser of the cut from Exercise 5 and separates the blocks almost perfectly here. For $k>2$ communities one stacks the first $k$
eigenvectors and runs k-means on the rows (spectral clustering, see [[Clustering]]); the normalised Laplacian is more robust to degree heterogeneity.

```python
L_sbm = np.diag(A_sbm.sum(1)) - A_sbm
w, V = np.linalg.eigh(L_sbm)
y_spec = (V[:, 1] > 0).astype(int)
acc_spec = max(np.mean(y_spec == y_sbm), np.mean(y_spec != y_sbm))
print('lambda_2 =', round(w[1], 3), ' accuracy =', acc_spec)
```

</details>

### Exercise 14 · Weisfeiler–Lehman colour refinement
*💻 Code · ★★★*

Implement 1-WL: all nodes start with colour 0; in each round a node's new colour is a hash of (own colour, **sorted multiset** of neighbour colours).
Return the histogram of colours after `k` rounds (use a colour dictionary **shared** between the graphs you compare so that colours are comparable).
Write `wl_same(A1, A2, k=3)` that says whether 1-WL *cannot distinguish* the two graphs. Test:
(a) `A_small` vs a randomly permuted copy, (b) a 6-cycle vs two disjoint triangles, (c) a path on 4 nodes vs a star on 4 nodes.

In [ ]:
def wl_same(A1, A2, k=3):
    # TODO: refine both graphs jointly; compare colour histograms after each round
    return None

perm = rng.permutation(6)
C6 = from_edges(6, [(i, (i + 1) % 6) for i in range(6)])
two_tri = from_edges(6, [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)])
P4 = from_edges(4, [(0, 1), (1, 2), (2, 3)])
S4 = from_edges(4, [(0, 1), (0, 2), (0, 3)])
wl_results = None  # [wl_same(A_small, permuted), wl_same(C6, two_tri), wl_same(P4, S4)]

check('WL results', wl_results, [True, True, False])

<details>
<summary><b>💡 Hint 1</b></summary>

Signature of node $v$: `(col[v], tuple(sorted(col[u] for u in neighbours(v))))`. Map each new signature to a new integer via a dict.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Two graphs are distinguished as soon as their colour histograms (Counter of colours) differ in some round.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Isomorphic graphs always get the same histograms → `True`. (b) **WL fails**: both graphs are 2-regular, so every node always sees the multiset $\{c, c\}$ and colours never split,
although one graph is connected and the other is not → `True` (indistinguishable). (c) The star's centre has degree 3 → distinguished after one round → `False`.

This is the test that bounds message-passing GNNs: GIN is exactly as powerful as 1-WL ([[Graph Neural Networks]]). The WL subtree **kernel** uses the concatenated histograms as feature vectors and takes dot products.

```python
from collections import Counter
def wl_same(A1, A2, k=3):
    graphs = [A1, A2]
    cols = [np.zeros(len(A), int) for A in graphs]
    for _ in range(k):
        table = {}
        new = []
        for A, c in zip(graphs, cols):
            sig = [(c[v], tuple(sorted(c[np.flatnonzero(A[v])]))) for v in range(len(A))]
            new.append(np.array([table.setdefault(s, len(table)) for s in sig]))
        cols = new
        if Counter(cols[0].tolist()) != Counter(cols[1].tolist()):
            return False
    return True

perm = rng.permutation(6)
C6 = from_edges(6, [(i, (i + 1) % 6) for i in range(6)])
two_tri = from_edges(6, [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)])
P4 = from_edges(4, [(0, 1), (1, 2), (2, 3)])
S4 = from_edges(4, [(0, 1), (0, 2), (0, 3)])
wl_results = [wl_same(A_small, A_small[perm][:, perm]), wl_same(C6, two_tri), wl_same(P4, S4)]
print(wl_results)
```

</details>

### Exercise 15 · DeepWalk as matrix factorisation
*💻 Code · ★★★*

Build a DeepWalk-style **shallow embedding** without a neural network:
1. From every node of `A_sbm` start 10 uniform random walks of length 20.
2. Count co-occurrences of node pairs within a window of 3 steps → matrix $M$.
3. Compute the positive PMI matrix $\text{PPMI}_{ij} = \max\big(0, \log\frac{M_{ij}\,\sum M}{M_{i\cdot}M_{\cdot j}}\big)$.
4. Take a rank-8 truncated SVD, embeddings $Z = U_8\sqrt{\Sigma_8}$.

Cluster $Z$ with `KMeans(2, n_init=10, random_state=0)` and store the accuracy (allowing swapped labels) in `acc_dw`.

In [ ]:
def random_walks(A, n_walks=10, length=20, seed=0):
    # TODO: return an int array of shape (n_nodes * n_walks, length)
    return None

acc_dw = None

check('DeepWalk-MF accuracy >= 0.9', None if acc_dw is None else acc_dw >= 0.9, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Neighbour lists `nbrs = [np.flatnonzero(A[i]) for i in range(n)]` make each walk step a `r.choice(nbrs[cur])`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For each walk and each offset $1\le w\le3$: `np.add.at(M, (W[:, :-w].ravel(), W[:, w:].ravel()), 1)`, then symmetrise $M \leftarrow M + M^\top$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Levy & Goldberg (2014) and Qiu et al. (2018) showed that skip-gram with negative sampling on random walks implicitly factorises a (shifted) PMI matrix of
walk co-occurrences, so DeepWalk ≈ matrix factorisation. Nodes in the same community co-occur on walks, get similar rows in PPMI and therefore nearby embeddings.
node2vec changes only the walk distribution (biased by $p, q$).

```python
def random_walks(A, n_walks=10, length=20, seed=0):
    r = np.random.default_rng(seed)
    n = len(A); nbrs = [np.flatnonzero(A[i]) for i in range(n)]
    W = np.zeros((n * n_walks, length), int)
    for k in range(n * n_walks):
        cur = k % n
        W[k, 0] = cur
        for t in range(1, length):
            cur = r.choice(nbrs[cur]) if len(nbrs[cur]) else cur
            W[k, t] = cur
    return W

W = random_walks(A_sbm)
n = len(A_sbm)
M = np.zeros((n, n))
for w in range(1, 4):
    np.add.at(M, (W[:, :-w].ravel(), W[:, w:].ravel()), 1)
M = M + M.T
with np.errstate(divide='ignore'):
    PPMI = np.maximum(0, np.log(M * M.sum() / np.outer(M.sum(1), M.sum(0))))
U, s, _ = np.linalg.svd(PPMI)
Z = U[:, :8] * np.sqrt(s[:8])
y_dw = KMeans(2, n_init=10, random_state=0).fit_predict(Z)
acc_dw = max(np.mean(y_dw == y_sbm), np.mean(y_dw != y_sbm))
print('accuracy:', acc_dw)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Graph ML Overview](Graph%20ML%20Overview.md).*